# Maximum Flow and Minimum Cut

This notebook is designed for live coding and student experimentation in a bachelor-level algorithms lecture.

It assumes students already know basic graph terminology, BFS/DFS, shortest-path algorithms, and the idea of cuts from minimum spanning trees. The main goal is to show that maximum flow is not just "another shortest path problem": we repeatedly search paths in a **residual graph**, update flow, and eventually read a minimum cut from the vertices still reachable from the source.

## Learning goals

- Model a directed capacity network with a source and a sink.
- Trace the Ford-Fulkerson method using Edmonds-Karp path selection.
- Explain residual capacity and reverse residual edges.
- Compute a maximum flow and extract a matching minimum cut.
- Verify a classroom implementation against NetworkX.

## Colab note

The setup cell installs `networkx` and `matplotlib` only if they are missing. The interactive slider uses `ipywidgets` when available; if widgets are not available, all demonstrations can still be explored by calling `show_step(result, step)` manually.


## From Shortest Paths to Flow Networks

Earlier graph lectures asked questions like "What is the cheapest path from `s` to `t`?" or "What is the shortest path from one source to all vertices?"

Maximum flow asks a different question:

> How much total material can we send from `s` to `t` at the same time if every directed edge has a capacity?

Useful comparisons:

| Earlier topic | Familiar idea | Flow version |
| --- | --- | --- |
| BFS | Explore reachable vertices layer by layer | Find an augmenting path in the residual graph |
| Dijkstra/A* | A path has a sum of edge costs | An augmenting path has a bottleneck: the minimum residual capacity on the path |
| Bellman-Ford | Repeatedly improve tentative values | Repeatedly improve a feasible flow |
| MST cut property | A cut partitions the vertices | A source-sink cut limits how much flow can ever pass |

The key new object is the **residual graph**. It contains every move that is still possible: unused forward capacity and the ability to cancel previous choices by using reverse residual edges.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from collections import deque

for package in ("networkx", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import matplotlib.pyplot as plt
import networkx as nx

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

plt.rcParams["figure.figsize"] = (8, 4.8)
plt.rcParams["axes.facecolor"] = "#fafafa"
plt.rcParams["font.size"] = 11

print(f"Python {sys.version.split()[0]}")
print(f"NetworkX {nx.__version__}")


## Problem Statement

A **flow network** is a directed graph where every edge `(u, v)` has a non-negative capacity `c(u, v)`.

A valid flow must obey two rules:

- **Capacity constraint**: `0 <= f(u, v) <= c(u, v)` for every original edge.
- **Flow conservation**: for every intermediate vertex, total inflow equals total outflow.

The **value** of a flow is the net amount leaving the source `s`.

A **cut** splits the vertices into two sets `S` and `T` where `s` is in `S` and `t` is in `T`. Its capacity is the sum of original capacities on edges crossing from `S` to `T`.

The **max-flow min-cut theorem** says:

> The value of the maximum flow is equal to the capacity of the minimum source-sink cut.


## Classroom Network

We will use one directed network for the main live coding demo. Think of edge capacities as bandwidth, road capacity, pipe volume, or number of students that can pass through a prerequisite/assignment pipeline per time unit.

The graph is small enough to trace by hand but large enough to need several augmenting paths.


In [ ]:
main_edges = [
    ("s", "a", 10),
    ("s", "b", 10),
    ("a", "b", 2),
    ("a", "c", 4),
    ("a", "d", 8),
    ("b", "d", 9),
    ("d", "c", 6),
    ("c", "t", 10),
    ("d", "t", 10),
]

main_pos = {
    "s": (0.0, 0.0),
    "a": (1.0, 0.9),
    "b": (1.0, -0.9),
    "c": (2.2, 0.9),
    "d": (2.2, -0.9),
    "t": (3.4, 0.0),
}


## Implementation: Residual Graph Helpers

The implementation below keeps one representation throughout the notebook:

- `capacity[(u, v)]` is the original edge capacity.
- `flow[(u, v)]` is the current flow on an original edge.
- `residual_capacity(u, v)` combines unused forward capacity and possible cancellation of flow on `(v, u)`.

This lets us demonstrate reverse residual edges without changing data structures halfway through the lecture.


In [ ]:
def normalize_capacity(edges_or_capacity):
    """Return a clean {(u, v): capacity} dictionary."""
    if isinstance(edges_or_capacity, dict):
        items = [(*edge, cap) for edge, cap in edges_or_capacity.items()]
    else:
        items = list(edges_or_capacity)

    capacity = {}
    for u, v, cap in items:
        if cap < 0:
            raise ValueError(f"Capacity must be non-negative, got {cap} on {u}->{v}")
        if u == v:
            raise ValueError("Self-loops are skipped in this teaching implementation")
        capacity[(u, v)] = cap
    return capacity


def nodes_from_capacity(capacity):
    nodes = set()
    for u, v in capacity:
        nodes.add(u)
        nodes.add(v)
    return sorted(nodes)


def adjacency_from_capacity(capacity):
    """Neighbors in the residual graph can go forward or backward on original edges."""
    adjacency = {node: [] for node in nodes_from_capacity(capacity)}
    for u, v in capacity:
        if v not in adjacency[u]:
            adjacency[u].append(v)
        if u not in adjacency[v]:
            adjacency[v].append(u)
    return {node: sorted(neighbors) for node, neighbors in adjacency.items()}


def empty_flow(capacity):
    return {edge: 0 for edge in capacity}


### Residual Capacity and BFS

This is the core algorithmic idea. BFS runs on the residual graph, not just on the original graph.


In [ ]:
def residual_capacity(capacity, flow, u, v):
    forward_unused = capacity.get((u, v), 0) - flow.get((u, v), 0)
    backward_flow = flow.get((v, u), 0)
    return forward_unused + backward_flow


def residual_edges(capacity, flow):
    adjacency = adjacency_from_capacity(capacity)
    residual = {}
    for u, neighbors in adjacency.items():
        for v in neighbors:
            remaining = residual_capacity(capacity, flow, u, v)
            if remaining > 0:
                residual[(u, v)] = remaining
    return residual


def find_augmenting_path_bfs(capacity, flow, source, sink):
    adjacency = adjacency_from_capacity(capacity)
    parent = {source: None}
    queue = deque([source])

    while queue and sink not in parent:
        u = queue.popleft()
        for v in adjacency[u]:
            if v not in parent and residual_capacity(capacity, flow, u, v) > 0:
                parent[v] = u
                queue.append(v)

    if sink not in parent:
        return [], 0

    path = []
    v = sink
    while v != source:
        u = parent[v]
        path.append((u, v))
        v = u
    path.reverse()

    bottleneck = min(residual_capacity(capacity, flow, u, v) for u, v in path)
    return path, bottleneck


def path_to_nodes(path):
    if not path:
        return []
    return [path[0][0]] + [v for _, v in path]


def path_to_text(path):
    if not path:
        return "(no path)"
    return " -> ".join(path_to_nodes(path))


### Updating Flow and Checking Invariants

Augmenting along a residual path either increases flow on a forward edge or cancels flow on the opposite original edge.


In [ ]:
def augment_flow(capacity, flow, path, amount):
    """Send amount through a residual path and return a new flow dictionary."""
    new_flow = dict(flow)
    for u, v in path:
        remaining = amount

        reverse_edge = (v, u)
        if reverse_edge in capacity and new_flow.get(reverse_edge, 0) > 0:
            cancelled = min(remaining, new_flow[reverse_edge])
            new_flow[reverse_edge] -= cancelled
            remaining -= cancelled

        if remaining > 0:
            if (u, v) not in capacity:
                raise ValueError(
                    f"Residual edge {u}->{v} can only cancel existing flow, "
                    "but there was not enough flow to cancel."
                )
            new_flow[(u, v)] = new_flow.get((u, v), 0) + remaining

        if (u, v) in capacity and new_flow.get((u, v), 0) > capacity[(u, v)]:
            raise AssertionError(f"Capacity exceeded on {u}->{v}")

    return new_flow


def flow_value(capacity, flow, source):
    outgoing = sum(value for (u, _), value in flow.items() if u == source)
    incoming = sum(value for (_, v), value in flow.items() if v == source)
    return outgoing - incoming


def reachable_in_residual_graph(capacity, flow, source):
    adjacency = adjacency_from_capacity(capacity)
    seen = {source}
    queue = deque([source])

    while queue:
        u = queue.popleft()
        for v in adjacency[u]:
            if v not in seen and residual_capacity(capacity, flow, u, v) > 0:
                seen.add(v)
                queue.append(v)
    return seen


def cut_edges_from_source_side(capacity, source_side):
    return [(u, v) for (u, v) in capacity if u in source_side and v not in source_side]


def cut_capacity(capacity, cut_edges):
    return sum(capacity[edge] for edge in cut_edges)


def verify_flow(capacity, flow, source, sink):
    errors = []
    for edge, cap in capacity.items():
        value = flow.get(edge, 0)
        if value < 0 or value > cap:
            errors.append(f"capacity violation on {edge}: {value}/{cap}")

    for node in nodes_from_capacity(capacity):
        if node in (source, sink):
            continue
        inflow = sum(value for (u, v), value in flow.items() if v == node)
        outflow = sum(value for (u, v), value in flow.items() if u == node)
        if inflow != outflow:
            errors.append(f"flow conservation violation at {node}: in={inflow}, out={outflow}")

    if errors:
        raise AssertionError("\n".join(errors))
    return True


### Edmonds-Karp Driver

The driver records snapshots before and after each augmentation so the same algorithm state can be visualized later.


In [ ]:
def make_snapshot(title, capacity, flow, path=None, bottleneck=None, note="", reachable=None, cut_edges=None):
    return {
        "title": title,
        "flow": dict(flow),
        "residual": residual_edges(capacity, flow),
        "path": list(path or []),
        "bottleneck": bottleneck,
        "note": note,
        "reachable": set(reachable) if reachable is not None else None,
        "cut_edges": list(cut_edges or []),
    }


def edmonds_karp(edges_or_capacity, source="s", sink="t"):
    capacity = normalize_capacity(edges_or_capacity)
    flow = empty_flow(capacity)
    snapshots = [make_snapshot("Initial state: all flows are zero", capacity, flow)]
    iteration = 0

    while True:
        path, bottleneck = find_augmenting_path_bfs(capacity, flow, source, sink)

        if not path:
            source_side = reachable_in_residual_graph(capacity, flow, source)
            cut_edges = cut_edges_from_source_side(capacity, source_side)
            snapshots.append(
                make_snapshot(
                    "No augmenting path remains: read the min cut",
                    capacity,
                    flow,
                    reachable=source_side,
                    cut_edges=cut_edges,
                    note=(
                        f"Reachable side S = {sorted(source_side)}. "
                        f"Cut capacity = {cut_capacity(capacity, cut_edges)}."
                    ),
                )
            )
            break

        iteration += 1
        snapshots.append(
            make_snapshot(
                f"Iteration {iteration}: BFS found an augmenting path",
                capacity,
                flow,
                path=path,
                bottleneck=bottleneck,
                note=f"Path {path_to_text(path)} has bottleneck {bottleneck}.",
            )
        )
        flow = augment_flow(capacity, flow, path, bottleneck)
        snapshots.append(
            make_snapshot(
                f"Iteration {iteration}: after sending {bottleneck}",
                capacity,
                flow,
                path=path,
                bottleneck=bottleneck,
                note=f"Current flow value = {flow_value(capacity, flow, source)}.",
            )
        )

    verify_flow(capacity, flow, source, sink)
    source_side = reachable_in_residual_graph(capacity, flow, source)
    final_cut_edges = cut_edges_from_source_side(capacity, source_side)
    return {
        "capacity": capacity,
        "flow": flow,
        "source": source,
        "sink": sink,
        "max_flow": flow_value(capacity, flow, source),
        "source_side": source_side,
        "sink_side": set(nodes_from_capacity(capacity)) - source_side,
        "cut_edges": final_cut_edges,
        "cut_capacity": cut_capacity(capacity, final_cut_edges),
        "snapshots": snapshots,
    }


## Visualization Helpers

Edge labels show `flow/capacity` on original directed edges.

Dashed green reverse arrows are **residual cancellation options**. They are not extra physical pipes. They mean: "we can undo this much flow on the opposite original edge if a later augmenting path needs it."


In [ ]:
def draw_flow_network(
    edges_or_capacity,
    flow=None,
    pos=None,
    path=None,
    reachable=None,
    cut_edges=None,
    title="",
    show_reverse_residual=True,
    ax=None,
):
    capacity = normalize_capacity(edges_or_capacity)
    flow = dict(flow or empty_flow(capacity))
    graph = nx.DiGraph()
    for (u, v), cap in capacity.items():
        graph.add_edge(u, v, capacity=cap)

    if pos is None:
        pos = nx.spring_layout(graph, seed=7)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4.8))

    path_edges = set(path or [])
    cut_edge_set = set(cut_edges or [])

    if reachable is None:
        node_colors = ["#f4e285" for _ in graph.nodes()]
    else:
        node_colors = ["#8ecae6" if node in reachable else "#ffdd99" for node in graph.nodes()]

    nx.draw_networkx_nodes(
        graph,
        pos,
        node_color=node_colors,
        edgecolors="#333333",
        node_size=1250,
        linewidths=1.4,
        ax=ax,
    )
    nx.draw_networkx_labels(graph, pos, font_weight="bold", ax=ax)

    edge_colors = []
    edge_widths = []
    edge_styles = []
    for edge in graph.edges():
        current = flow.get(edge, 0)
        cap = capacity[edge]
        if edge in cut_edge_set:
            edge_colors.append("#7b2cbf")
            edge_widths.append(4.0)
            edge_styles.append("solid")
        elif edge in path_edges:
            edge_colors.append("#d1495b")
            edge_widths.append(4.0)
            edge_styles.append("solid")
        elif current == cap:
            edge_colors.append("#555555")
            edge_widths.append(2.8)
            edge_styles.append("solid")
        else:
            edge_colors.append("#9aa0a6")
            edge_widths.append(2.0)
            edge_styles.append("solid")

    nx.draw_networkx_edges(
        graph,
        pos,
        edge_color=edge_colors,
        width=edge_widths,
        style=edge_styles,
        arrows=True,
        arrowsize=18,
        min_source_margin=18,
        min_target_margin=18,
        connectionstyle="arc3,rad=0.08",
        ax=ax,
    )

    edge_labels = {(u, v): f"{flow.get((u, v), 0)}/{cap}" for (u, v), cap in capacity.items()}
    nx.draw_networkx_edge_labels(
        graph,
        pos,
        edge_labels=edge_labels,
        rotate=False,
        font_size=10,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.78, "pad": 0.2},
        ax=ax,
    )

    if show_reverse_residual:
        reverse_edges = []
        reverse_labels = {}
        for (u, v), current_flow in flow.items():
            if current_flow > 0:
                reverse_edges.append((v, u))
                reverse_labels[(v, u)] = str(current_flow)
        if reverse_edges:
            nx.draw_networkx_edges(
                graph,
                pos,
                edgelist=reverse_edges,
                edge_color="#2a9d8f",
                width=1.6,
                style="dashed",
                arrows=True,
                arrowsize=14,
                min_source_margin=22,
                min_target_margin=22,
                connectionstyle="arc3,rad=-0.23",
                ax=ax,
            )
            nx.draw_networkx_edge_labels(
                graph,
                pos,
                edge_labels=reverse_labels,
                rotate=False,
                font_size=9,
                font_color="#1b6f5f",
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.65, "pad": 0.1},
                ax=ax,
            )

    if title:
        ax.set_title(title)
    ax.axis("off")
    return ax


### Displaying Snapshots

`show_step` is intentionally separate from the algorithm. This keeps the algorithm testable and lets us reuse the same snapshots with a slider or with manual calls.


In [ ]:
def print_flow_table(capacity, flow):
    print("edge    flow/capacity    residual forward")
    for u, v in sorted(capacity):
        print(f"{u:>2} -> {v:<2}   {flow.get((u, v), 0):>2}/{capacity[(u, v)]:<2}             {residual_capacity(capacity, flow, u, v):>2}")


def show_step(result, step, pos=None, show_reverse_residual=True):
    snapshots = result["snapshots"]
    step = max(0, min(int(step), len(snapshots) - 1))
    snapshot = snapshots[step]

    draw_flow_network(
        result["capacity"],
        flow=snapshot["flow"],
        pos=pos,
        path=snapshot["path"],
        reachable=snapshot["reachable"],
        cut_edges=snapshot["cut_edges"],
        title=f"Step {step}: {snapshot['title']}",
        show_reverse_residual=show_reverse_residual,
    )
    plt.show()

    print(snapshot["title"])
    if snapshot["path"]:
        print(f"augmenting path: {path_to_text(snapshot['path'])}")
        print(f"bottleneck: {snapshot['bottleneck']}")
    if snapshot["note"]:
        print(snapshot["note"])
    print(f"flow value: {flow_value(result['capacity'], snapshot['flow'], result['source'])}")
    if snapshot["cut_edges"]:
        print(f"cut edges: {snapshot['cut_edges']}")


## Draw the Initial Network

Before running the algorithm, inspect the capacities. Every label is currently `0/capacity`.


In [ ]:
main_capacity = normalize_capacity(main_edges)
draw_flow_network(main_capacity, pos=main_pos, title="Initial capacity network", show_reverse_residual=False)
plt.show()


## Run Edmonds-Karp

Edmonds-Karp is the Ford-Fulkerson method with a specific path choice: use BFS to find an augmenting path with the fewest edges in the current residual graph.

That path is not necessarily the path with the largest bottleneck. The value of BFS here is predictability and the `O(VE^2)` worst-case bound.


In [ ]:
result = edmonds_karp(main_capacity, source="s", sink="t")

print(f"max flow = {result['max_flow']}")
print(f"min cut capacity = {result['cut_capacity']}")
print(f"source side of final residual graph = {sorted(result['source_side'])}")
print(f"sink side = {sorted(result['sink_side'])}")
print(f"cut edges = {result['cut_edges']}")
print()
print_flow_table(result["capacity"], result["flow"])


In [ ]:
show_step(result, 0, pos=main_pos)


## Step-by-Step Exploration

Run the next cell in Colab or Jupyter to browse algorithm snapshots with a slider.

If the widget does not appear, use manual calls such as:

```python
show_step(result, 4, pos=main_pos)
show_step(result, len(result["snapshots"]) - 1, pos=main_pos)
```


In [ ]:
try:
    from IPython import get_ipython
    from ipywidgets import Checkbox, IntSlider, interact
    notebook_ui = get_ipython() is not None
except Exception:
    Checkbox = None
    IntSlider = None
    interact = None
    notebook_ui = False

if interact is not None and notebook_ui:
    @interact(
        step=IntSlider(min=0, max=len(result["snapshots"]) - 1, step=1, value=0),
        show_reverse_residual=Checkbox(value=True, description="reverse residual edges"),
    )
    def browse_edmonds_karp(step=0, show_reverse_residual=True):
        show_step(result, step, pos=main_pos, show_reverse_residual=show_reverse_residual)
else:
    print("ipywidgets is not available here. Use show_step(result, step, pos=main_pos) manually.")


## Max-Flow Min-Cut Check

After Edmonds-Karp stops, there is no residual path from `s` to `t`.

The vertices still reachable from `s` in the final residual graph form the source side `S` of a minimum cut. Every original edge from `S` to `T` is saturated; otherwise we could cross it in the residual graph.


In [ ]:
final_step = len(result["snapshots"]) - 1
show_step(result, final_step, pos=main_pos, show_reverse_residual=False)

assert result["max_flow"] == result["cut_capacity"]
print(f"max flow equals min cut capacity: {result['max_flow']} == {result['cut_capacity']}")


## Why Reverse Residual Edges Matter

Reverse residual edges are usually the concept that separates "I can run the code" from "I understand the algorithm".

In the small example below, suppose a poor first path sends flow through `s -> a -> b -> t`. That uses the middle edge `a -> b`. To reach the true maximum flow of 2, the next augmenting path must use the reverse residual edge `b -> a`, which cancels the earlier choice on `a -> b`.

This is not sending material backward in the real network. It is correcting the bookkeeping of a previous path choice.


In [ ]:
reroute_edges = [
    ("s", "a", 1),
    ("s", "b", 1),
    ("a", "b", 1),
    ("a", "t", 1),
    ("b", "t", 1),
]
reroute_capacity = normalize_capacity(reroute_edges)
reroute_pos = {
    "s": (0.0, 0.0),
    "a": (1.0, 0.8),
    "b": (1.0, -0.8),
    "t": (2.0, 0.0),
}

starting_flow = empty_flow(reroute_capacity)
poor_first_path = [("s", "a"), ("a", "b"), ("b", "t")]
flow_after_poor_choice = augment_flow(reroute_capacity, starting_flow, poor_first_path, 1)

repair_path, repair_bottleneck = find_augmenting_path_bfs(
    reroute_capacity,
    flow_after_poor_choice,
    "s",
    "t",
)
repaired_flow = augment_flow(reroute_capacity, flow_after_poor_choice, repair_path, repair_bottleneck)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
draw_flow_network(
    reroute_capacity,
    flow_after_poor_choice,
    pos=reroute_pos,
    path=poor_first_path,
    title="After poor first path",
    show_reverse_residual=True,
    ax=axes[0],
)
draw_flow_network(
    reroute_capacity,
    repaired_flow,
    pos=reroute_pos,
    path=repair_path,
    title="After repair path uses b -> a residual edge",
    show_reverse_residual=True,
    ax=axes[1],
)
plt.show()

print(f"poor first path: {path_to_text(poor_first_path)}")
print(f"repair path in residual graph: {path_to_text(repair_path)}")
print(f"repair bottleneck: {repair_bottleneck}")
print(f"final flow value: {flow_value(reroute_capacity, repaired_flow, 's')}")
print_flow_table(reroute_capacity, repaired_flow)


## Cross-Check with NetworkX

Classroom implementations are easier to trust when we compare them with a mature graph library.


In [ ]:
def networkx_flow_check(capacity, source="s", sink="t", expected_result=None):
    graph = nx.DiGraph()
    for (u, v), cap in capacity.items():
        graph.add_edge(u, v, capacity=cap)

    nx_flow_value, nx_flow_dict = nx.maximum_flow(graph, source, sink, capacity="capacity")
    nx_cut_value, nx_partition = nx.minimum_cut(graph, source, sink, capacity="capacity")

    print(f"NetworkX maximum flow value: {nx_flow_value}")
    print(f"NetworkX minimum cut value: {nx_cut_value}")
    print(f"NetworkX cut partition: S={sorted(nx_partition[0])}, T={sorted(nx_partition[1])}")

    if expected_result is not None:
        assert nx_flow_value == expected_result["max_flow"]
        assert nx_cut_value == expected_result["cut_capacity"]
    return nx_flow_dict


nx_flow_dict = networkx_flow_check(main_capacity, "s", "t", expected_result=result)


## Complexity and Algorithm Choices

The Ford-Fulkerson method is the general idea: repeatedly find an augmenting path in the residual graph and augment along it.

Common variants:

| Algorithm | Path choice / main idea | Typical worst-case bound |
| --- | --- | --- |
| Ford-Fulkerson with arbitrary paths | Any augmenting path | `O(E * |f*|)` for integral capacities, where `|f*|` is the max-flow value |
| Edmonds-Karp | BFS path with fewest edges | `O(VE^2)` |
| Dinic | BFS levels plus blocking flows | `O(V^2E)` in general graphs |
| Push-relabel | Maintains a preflow and pushes excess locally | Often very fast in practice; several variants exist |

For this lecture, Edmonds-Karp is the right main implementation because it reuses BFS and exposes the residual graph clearly. Dinic and push-relabel are excellent follow-up topics, but they should come after students are comfortable with residual capacity and min-cut extraction.


## Experiments for Students

Try these changes and rerun the notebook from the classroom network cell onward:

1. Change `("b", "d", 9)` to capacity `3`. Which cut becomes minimum?
2. Add a new edge `("b", "c", 5)`. Does the maximum flow increase?
3. Change the sink edges so `("c", "t", 10)` and `("d", "t", 10)` become very large. What becomes the bottleneck?
4. Add an edge from `a` directly to `t`. Predict the new max flow before running the code.
5. Modify `find_augmenting_path_bfs` to use a stack instead of a queue. How does the path sequence change?

Good debugging questions:

- Which edges are saturated in the final graph?
- Which vertices are reachable from `s` after the algorithm stops?
- Does every intermediate vertex have equal inflow and outflow?
- Can you explain every dashed reverse residual edge as an undo option?


## Summary

Main takeaways:

- A flow network has capacities, not distances.
- An augmenting path is useful only up to its bottleneck residual capacity.
- Reverse residual edges let later paths cancel earlier decisions.
- Edmonds-Karp is Ford-Fulkerson with BFS path selection.
- When no augmenting path remains, the vertices reachable from `s` in the residual graph define a minimum cut.
- The final flow value equals the capacity of that cut.

## Further Reading

- Cormen, Leiserson, Rivest, Stein, *Introduction to Algorithms*, 4th edition, Chapter 24.
- Jeff Erickson, *Algorithms*, Chapter 10, Maximum Flows and Minimum Cuts: https://jeffe.cs.illinois.edu/teaching/algorithms/
- NetworkX maximum flow documentation: https://networkx.org/documentation/stable/reference/algorithms/flow.html
- L. R. Ford Jr. and D. R. Fulkerson, "Maximal Flow Through a Network" (1956).
- Jack Edmonds and Richard Karp, "Theoretical Improvements in Algorithmic Efficiency for Network Flow Problems" (1972).
